# 🚀 50M Bengali GPT - ২-স্টেজ প্রোডাকশন ট্রেনিং (Free Colab Optimized)
### 🧠 ২-স্টেজ আর্কিটেকচার:
1. **স্টেজ ১ (Pretraining - ভাষা শিক্ষা):** `corpus.txt` (~৩,৫০,০০০+ লাইন উইকিপিডিয়া ও জ্ঞান) পড়ে মডেল ব্যাকরণ, শব্দভাণ্ডার ও বাক্যগঠন শেখে (২ ইপক, lr=3e-4)।
2. **স্টেজ ২ (SFT - প্রশ্ন-উত্তর ও চ্যাটবট শিক্ষা):** `sft_data.txt` (~১,৫০,০০০+ লাইন বাংলা ইনস্ট্রাকশন ও গণিত) পড়ে মডেল সুনির্দিষ্ট উত্তর দেওয়া শেখে (৩ ইপক, lr=1e-4)।

### ⚡ সর্বোচ্চ গতি ও সুরক্ষা (PyTorch 2.0):
- **PyTorch Speed:** SDPA FlashAttention + AMP FP16 + PyTorch 2.0 `torch.compile` (সর্বোচ্চ স্পিড!)
- **Auto-Resume:** সেশন কেটে গেলেও পুনরায় রান করলে শেষ সেভ হওয়া স্টেপ থেকে নিজে নিজেই শুরু হবে (জিরো ডেটা লস)।
- **প্যারামিটার:** ~54.3 Million (১০০% আনফ্রোজেন, কোনো LoRA/QLoRA ছাড়া সম্পূর্ণ মডেল স্ক্র্যাচ থেকে শিখবে)
- **কনটেক্সট লেন্থ:** 512 Tokens (~৩০০-৩৫০ বাংলা শব্দ)
- **ভোকাবুলারি:** 10,000 (ByteLevel BPE - বাংলা, ইংরেজি ও গণিত সমৃদ্ধ)
- **হার্ডওয়্যার:** Colab Free T4 GPU (~1.8 GB VRAM খরচ, 14 GB মেমরি নিরাপদ)

In [ ]:
# Step 1: GPU চেক করুন (NVIDIA T4 থাকা নিশ্চিত করুন)
!nvidia-smi

In [ ]:
# Step 2: Google Drive মাউন্ট করুন (উভয় স্টেজের চেকপয়েন্ট সেভ করার জন্য)
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_BASE_DIR = '/content/drive/MyDrive/bengali_gpt_50m_checkpoints'
STAGE1_DIR = os.path.join(DRIVE_BASE_DIR, 'stage1_pretrain')
STAGE2_DIR = os.path.join(DRIVE_BASE_DIR, 'stage2_sft')
os.makedirs(STAGE1_DIR, exist_ok=True)
os.makedirs(STAGE2_DIR, exist_ok=True)
print(f"✓ Drive ডিরেক্টরি রেডি:\n  - Stage 1: {STAGE1_DIR}\n  - Stage 2: {STAGE2_DIR}")

In [ ]:
# Step 3: রিপোজিটরি ক্লোন ও ডিপেনডেন্সি ইনস্টল করুন
import os

%cd /content
!rm -rf ss_100m
!git clone https://github.com/kajshikhi49-afk/ss_100m.git

%cd /content/ss_100m/ss_50million
!pip install -q -r requirements.txt
print("✓ এনভায়রনমেন্ট ও ডিপেনডেন্সি প্রস্তুত!")

In [ ]:
# Step 4: 📁 বৃহৎ ডেটাসেট প্রস্তুতকরণ (Corpus: ~৩.৫ লাখ লাইন | SFT: ~১.৫ লাখ লাইন)
import os
import re
import json
import random
from datasets import load_dataset

os.makedirs('data', exist_ok=True)
corpus_path = 'data/corpus.txt'      # স্টেজ-১ এর জন্য
sft_data_path = 'data/sft_data.txt'  # স্টেজ-২ এর জন্য

need_download = True
if os.path.exists(corpus_path) and os.path.exists(sft_data_path):
    with open(corpus_path, 'r', encoding='utf-8') as f:
        line_cnt_c = sum(1 for _ in f)
    with open(sft_data_path, 'r', encoding='utf-8') as f:
        line_cnt_s = sum(1 for _ in f)
    if line_cnt_c >= 300000 and line_cnt_s >= 100000:
        print(f"✓ বিদ্যমান বৃহৎ কর্পাস ফাইল পাওয়া গেছে (Corpus: {line_cnt_c:,} | SFT: {line_cnt_s:,}), স্কিপ করা হলো।")
        need_download = False
    else:
        print(f"⚠️ ফাইল সাইজ নতুন বৃহৎ টার্গেটের চেয়ে ছোট! ফ্রেশ এক্সপ্যান্ডেড ডাউনলোড শুরু হচ্ছে...")

if need_download:
    print("⚡ বিশাল মাল্টি-সোর্স ডেটা ডাউনলোড ও প্রসেসিং শুরু হচ্ছে...")
    corpus_lines = []
    sft_lines = []

    # ১. লোকাল বা ড্রাইভের বিশেষায়িত ডোমেন ডেটা
    jsonl_files = ['data/digital_marketing.jsonl', '/content/drive/MyDrive/digital_marketing.jsonl', 'data/nctb_data.jsonl']
    for jf in jsonl_files:
        if os.path.exists(jf):
            print(f"  -> পাওয়া গেছে: {jf}")
            with open(jf, 'r', encoding='utf-8') as f:
                for line in f:
                    try:
                        d_json = json.loads(line.strip())
                        inst = d_json.get('instruction', '').strip()
                        out = d_json.get('output', '').strip()
                        if inst and out:
                            corpus_lines.append(f"{inst} {out}")
                            sft_lines.append(f"প্রশ্ন: {inst} উত্তর: {out} <EOS>")
                    except:
                        pass

    # ২. বাংলা উইকিপিডিয়া সংগ্রহ (২,৫০,০০০+ লাইন - Corpus এর মূল মেরুদণ্ড)
    print("  -> বাংলা উইকিপিডিয়া সংগ্রহ হচ্ছে (২,৫০,০০০ লাইন)...")
    wiki_bn = load_dataset('wikimedia/wikipedia', '20231101.bn', split='train', streaming=True)
    bn_count = 0
    for item in wiki_bn:
        for p in item.get('text', '').split('\n'):
            p = p.strip()
            if len(p) >= 30 and re.search(r'[\u0980-\u09FF]', p):
                corpus_lines.append(p)
                bn_count += 1
                if bn_count >= 250000: break
        if bn_count >= 250000: break
    print(f"     ✓ সংগৃহীত বাংলা উইকিপিডিয়া লাইন: {bn_count:,}")

    # ৩. ইংরেজি উইকিপিডিয়া সংগ্রহ (৬০,০০০ লাইন)
    print("  -> ইংরেজি উইকিপিডিয়া সংগ্রহ হচ্ছে (৬০,০০০ লাইন)...")
    wiki_en = load_dataset('wikimedia/wikipedia', '20231101.en', split='train', streaming=True)
    en_count = 0
    for item in wiki_en:
        for p in item.get('text', '').split('\n'):
            p = p.strip()
            if len(p) >= 35 and re.search(r'[a-zA-Z]', p):
                corpus_lines.append(p)
                en_count += 1
                if en_count >= 60000: break
        if en_count >= 60000: break
    print(f"     ✓ সংগৃহীত ইংরেজি উইকিপিডিয়া লাইন: {en_count:,}")

    # ৪. বাংলা ইনস্ট্রাকশন / চ্যাট ডেটাসেট (SFT এর মূল ভিত্তি - ৫০,০০০+ প্রম্পট)
    print("  -> বাংলা ইনস্ট্রাকশন (Alpaca / Orca) ডেটাসেট সংগ্রহ হচ্ছে...")
    try:
        alpaca_ds = load_dataset('BanglaLLM/bangla-alpaca-orca', split='train', streaming=True)
        alp_cnt = 0
        for row in alpaca_ds:
            inst = row.get('instruction', '').strip()
            inp = row.get('input', '').strip()
            out = row.get('output', '').strip()
            if inst and out:
                full_q = f"{inst} {inp}".strip()
                sft_lines.append(f"প্রশ্ন: {full_q} উত্তর: {out} <EOS>")
                corpus_lines.append(f"{full_q} {out}")
                alp_cnt += 1
                if alp_cnt >= 50000: break
        print(f"     ✓ সংগৃহীত বাংলা ইনস্ট্রাকশন জোড়া: {alp_cnt:,}")
    except Exception as e:
        print(f"     ⚠️ Alpaca ডাউনলোড ফলব্যাক: {e}")

    # ৫. পাটিগণিত, শতকরা, অনুপাত ও জেনারেল গণিত সমস্যা তৈরি (৫০,০০০ লাইন)
    print("  -> পাটিগণিত, ঐকিক নিয়ম ও জেনারেল গণিত তৈরি হচ্ছে (৫০,০০০ লাইন)...")
    random.seed(42)
    for _ in range(50000):
        m_type = random.choice(['add', 'sub', 'mul', 'div', 'percent', 'unitary', 'en_math'])
        if m_type == 'add':
            a, b = random.randint(5, 999), random.randint(5, 999)
            q = f"{a} এর সাথে {b} যোগ করলে কত হয়?"
            ans = f"{a} + {b} = {a + b}।"
        elif m_type == 'sub':
            a, b = random.randint(50, 999), random.randint(5, 500)
            if a < b: a, b = b, a
            q = f"{a} থেকে {b} বিয়োগ করলে কত অবশিষ্ট থাকে?"
            ans = f"{a} - {b} = {a - b}।"
        elif m_type == 'mul':
            a, b = random.randint(2, 99), random.randint(2, 50)
            q = f"{a} কে {b} দিয়ে গুণ করলে কত হবে?"
            ans = f"{a} × {b} = {a * b}।"
        elif m_type == 'div':
            divisor = random.randint(2, 25)
            quotient = random.randint(2, 50)
            dividend = divisor * quotient
            q = f"{dividend} কে {divisor} দিয়ে ভাগ করলে ভাগফল কত হয়?"
            ans = f"{dividend} ÷ {divisor} = {quotient}।"
        elif m_type == 'percent':
            base = random.choice([50, 100, 200, 300, 500, 1000, 2000])
            rate = random.choice([5, 10, 15, 20, 25, 50])
            val = int(base * (rate / 100))
            q = f"{base} টাকার {rate}% কত টাকা?"
            ans = f"{base} টাকার {rate}% হলো {val} টাকা।"
        elif m_type == 'unitary':
            n1 = random.randint(2, 6)
            unit_price = random.randint(5, 40)
            cost1 = n1 * unit_price
            n2 = random.randint(7, 15)
            cost2 = n2 * unit_price
            q = f"যদি {n1}টি ডিমের দাম {cost1} টাকা হয়, তবে {n2}টি ডিমের দাম কত?"
            ans = f"১টি ডিমের দাম {cost1} ÷ {n1} = {unit_price} টাকা। সুতরাং {n2}টি ডিমের দাম {unit_price} × {n2} = {cost2} টাকা।"
        else:
            m1, m2 = random.randint(2, 50), random.randint(2, 25)
            q = f"What is {m1} multiplied by {m2}?"
            ans = f"{m1} * {m2} = {m1 * m2}."

        corpus_lines.append(f"{q} {ans}")
        sft_lines.append(f"প্রশ্ন: {q} উত্তর: {ans} <EOS>")

    print("⚡ ডেটা সুষমভাবে এলোমেলো (Shuffle) করা হচ্ছে...")
    random.shuffle(corpus_lines)
    random.shuffle(sft_lines)

    with open(corpus_path, 'w', encoding='utf-8') as f:
        for l in corpus_lines:
            f.write(l + '\n')

    with open(sft_data_path, 'w', encoding='utf-8') as f:
        for l in sft_lines:
            f.write(l + '\n')

    # পুরাতন ক্যাশ মুছে নতুন বিশাল ডেটা ফ্রেশ টোকেনাইজ নিশ্চিত করা
    for old_bin in ['data/corpus_tokens.bin', 'data/sft_data_tokens.bin']:
        if os.path.exists(old_bin):
            os.remove(old_bin)

    print("=" * 65)
    print(f"✓ বৃহৎ স্টেজ-১ ফাইল (corpus.txt): {len(corpus_lines):,} লাইন ({os.path.getsize(corpus_path)/(1024*1024):.1f} MB)")
    print(f"✓ বৃহৎ স্টেজ-২ ফাইল (sft_data.txt): {len(sft_lines):,} লাইন ({os.path.getsize(sft_data_path)/(1024*1024):.1f} MB)")
    print("=" * 65)


In [ ]:
# Step 5: ⚡ সমন্বিত কর্পাস থেকে মাল্টিলিঙ্গুয়াল 10,000 Vocab BPE টোকেনাইজার তৈরি
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

special_tokens = ['<PAD>', '<UNK>', '<BOS>', '<EOS>', '<|system|>', '<|user|>', '<|assistant|>', '<|math|>']
tok = Tokenizer(models.BPE(unk_token='<UNK>'))
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=False)
tok.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=10000,
    special_tokens=special_tokens,
    min_frequency=2,
    show_progress=True
)

print("⚡ নতুন বৃহৎ ডেটায় টোকেনাইজার ট্রেনিং শুরু হচ্ছে...")
tok.train(['data/corpus.txt', 'data/sft_data.txt'], trainer)
tok.save('tokenizer.json')
print(f"✓ টোকেনাইজার প্রস্তুত! Vocab Size: {tok.get_vocab_size():,}")

In [ ]:
# Step 6: 🚀 [স্টেজ ১] প্রি-ট্রেনিং (Pretraining - ভাষা শিক্ষা | ২ ইপক)
# Auto-Resume সক্রিয় + PyTorch 2.0 সর্বোচ্চ গতি অপটিমাইজেশন
import os
import sys
import glob
import time
import math
import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler
from src.config import GPTConfig
from src.model import BengaliGPT as GPT
from src.dataset import BengaliDataset
from tokenizers import Tokenizer

# ১. ডিভাইস ও সর্বোচ্চ GPU গতি (FlashAttention + TF32)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

# ২. ডেটাসেট লোড (corpus.txt)
tokenizer = Tokenizer.from_file("tokenizer.json")
dataset_stage1 = BengaliDataset(corpus_path="data/corpus.txt", tokenizer=tokenizer, block_size=GPTConfig.block_size, split_ratio=0.9)

tokens_per_step = GPTConfig.batch_size * GPTConfig.gradient_accumulation_steps * GPTConfig.block_size
steps_per_epoch = dataset_stage1.train_len // tokens_per_step
STAGE1_EPOCHS = 2.0
stage1_max_iters = int(steps_per_epoch * STAGE1_EPOCHS)
print(f"✓ স্টেজ-১ লক্ষ্য: {STAGE1_EPOCHS} ইপক = {stage1_max_iters:,} স্টেপস")

# ৩. মডেল ইনিশিয়ালাইজেশন ও PyTorch 2.0 স্পিড কম্পাইলেশন
raw_model = GPT(GPTConfig).to(device)
try:
    model = torch.compile(raw_model)
    print("✓ PyTorch 2.0 torch.compile সক্রিয় (সর্বোচ্চ স্পিড)!")
except Exception:
    model = raw_model

optimizer = torch.optim.AdamW(raw_model.parameters(), lr=GPTConfig.learning_rate, betas=(0.9, 0.95), weight_decay=0.1)
scaler = GradScaler()

# ৪. Auto-Resume চেক
start_step = 1
existing_ckpts = glob.glob(os.path.join(STAGE1_DIR, "stage1_step_*.pt"))
if existing_ckpts:
    def get_step_num(p): 
        try: return int(p.split('_step_')[-1].replace('.pt', ''))
        except: return 0
    latest_ckpt = sorted(existing_ckpts, key=get_step_num)[-1]
    ckpt_step = get_step_num(latest_ckpt)
    if ckpt_step > 0 and ckpt_step < stage1_max_iters:
        print(f"📥 [AUTO-RESUME] শেষ চেকপয়েন্ট পাওয়া গেছে: {latest_ckpt}")
        raw_model.load_state_dict(torch.load(latest_ckpt, map_location=device))
        start_step = ckpt_step + 1
        print(f"🔄 স্টেজ ১ পুনরায় শুরু হচ্ছে স্টেপ {start_step:,} থেকে!")

def get_lr(it, max_iters, lr=3e-4, min_lr=3e-5):
    warmup = 250
    if it < warmup: return lr * it / warmup
    if it > max_iters: return min_lr
    decay = (it - warmup) / (max_iters - warmup)
    return min_lr + 0.5 * (1.0 + math.cos(math.pi * decay)) * (lr - min_lr)

print("=" * 65)
print(f"🔥 [স্টেজ ১] প্রি-ট্রেনিং চলছে ({start_step:,} -> {stage1_max_iters:,} স্টেপস)... ")
print("=" * 65)

model.train()
optimizer.zero_grad(set_to_none=True)
start_time = time.time()

for step in range(start_step, stage1_max_iters + 1):
    lr = get_lr(step, stage1_max_iters, lr=GPTConfig.learning_rate, min_lr=GPTConfig.min_lr)
    for param_group in optimizer.param_groups: param_group['lr'] = lr

    accum_loss = 0.0
    for micro in range(GPTConfig.gradient_accumulation_steps):
        x, y = dataset_stage1.get_batch('train', batch_size=GPTConfig.batch_size, device=device)
        with autocast(dtype=torch.float16):
            _, loss = model(x, targets=y)  # VRAM সুরক্ষিত ও ফাস্ট
            loss = loss / GPTConfig.gradient_accumulation_steps
        scaler.scale(loss).backward()
        accum_loss += loss.item()

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(raw_model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)

    if step % 250 == 0 or step == start_step:
        ep = (step * tokens_per_step) / dataset_stage1.train_len
        elapsed = time.time() - start_time
        spd = (step - start_step + 1) / elapsed if elapsed > 0 else 0
        print(f"[Stage 1] Step {step:5d}/{stage1_max_iters} (Epoch {ep:.2f}) | Loss: {accum_loss:.4f} | LR: {lr:.2e} | Speed: {spd:.2f} it/s")

    if step % 500 == 0 or step == stage1_max_iters:
        ckpt = os.path.join(STAGE1_DIR, f"stage1_step_{step}.pt")
        torch.save(raw_model.state_dict(), ckpt)

# স্টেজ ১ ফাইনাল সেভ
stage1_final_path = os.path.join(DRIVE_BASE_DIR, "checkpoint_stage_1.pt")
torch.save(raw_model.state_dict(), stage1_final_path)
print(f"🎉 স্টেজ ১ সম্পন্ন! মডেল সেভ হয়েছে: {stage1_final_path}")

In [ ]:
# Step 7: 🎯 [স্টেজ ২] SFT ফাইন-টিউনিং (বৃহৎ চ্যাটবট শিক্ষা | ৩ ইপক)
import os
import glob
import time
import math
import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler
from src.config import GPTConfig
from src.model import BengaliGPT as GPT
from src.dataset import BengaliDataset
from tokenizers import Tokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# ১. স্টেজ ১ এর ফাইনাল মডেল লোড করা
stage1_ckpt_path = os.path.join(DRIVE_BASE_DIR, "checkpoint_stage_1.pt")
if not os.path.exists(stage1_ckpt_path):
    stage1_ckpt_path = "checkpoints/stage_1/stage_1_final.pt"

raw_sft = GPT(GPTConfig).to(device)
if os.path.exists(stage1_ckpt_path):
    print(f"📥 স্টেজ ১ মডেল লোড হয়েছে: {stage1_ckpt_path}")
    raw_sft.load_state_dict(torch.load(stage1_ckpt_path, map_location=device))
else:
    print("⚠️ সতর্কতা: স্টেজ ১ চেকপয়েন্ট পাওয়া যায়নি, স্ক্র্যাচ থেকে শুরু হচ্ছে!")

try:
    sft_model = torch.compile(raw_sft)
    print("✓ PyTorch 2.0 torch.compile সক্রিয়!")
except Exception:
    sft_model = raw_sft

# ২. স্টেজ ২ ডেটাসেট লোড (sft_data.txt)
tokenizer = Tokenizer.from_file("tokenizer.json")
dataset_stage2 = BengaliDataset(corpus_path="data/sft_data.txt", tokenizer=tokenizer, block_size=GPTConfig.block_size, split_ratio=0.9)

tokens_per_step = GPTConfig.batch_size * GPTConfig.gradient_accumulation_steps * GPTConfig.block_size
steps_per_epoch = max(1, dataset_stage2.train_len // tokens_per_step)
STAGE2_EPOCHS = 3.0  # বৃহৎ ১.৫ লাখ ডেটাসেটের জন্য ৩ ইপক যথেষ্ট শক্তিশালী
stage2_max_iters = int(steps_per_epoch * STAGE2_EPOCHS)
print(f"✓ স্টেজ-২ লক্ষ্য: {STAGE2_EPOCHS} ইপক = {stage2_max_iters:,} স্টেপস")

# ৩. অপটিমাইজার (লোয়ার লার্নিং রেট 1e-4)
optimizer = torch.optim.AdamW(raw_sft.parameters(), lr=GPTConfig.sft_learning_rate, betas=(0.9, 0.95), weight_decay=0.1)
scaler = GradScaler()

# ৪. Auto-Resume চেক
start_step_sft = 1
existing_sft = glob.glob(os.path.join(STAGE2_DIR, "stage2_step_*.pt"))
if existing_sft:
    def get_sft_num(p):
        try: return int(p.split('_step_')[-1].replace('.pt', ''))
        except: return 0
    latest_sft = sorted(existing_sft, key=get_sft_num)[-1]
    s_step = get_sft_num(latest_sft)
    if s_step > 0 and s_step < stage2_max_iters:
        print(f"📥 [AUTO-RESUME] স্টেজ ২ চেকপয়েন্ট পাওয়া গেছে: {latest_sft}")
        raw_sft.load_state_dict(torch.load(latest_sft, map_location=device))
        start_step_sft = s_step + 1
        print(f"🔄 স্টেজ ২ পুনরায় শুরু হচ্ছে স্টেপ {start_step_sft:,} থেকে!")

def get_sft_lr(it, max_iters, lr=1e-4, min_lr=1e-5):
    warmup = min(100, max_iters // 10)
    if it < warmup: return lr * it / warmup
    if it > max_iters: return min_lr
    decay = (it - warmup) / max(1, (max_iters - warmup))
    return min_lr + 0.5 * (1.0 + math.cos(math.pi * decay)) * (lr - min_lr)

print("=" * 65)
print(f"🔥 [স্টেজ ২] SFT চ্যাটবট ট্রেনিং চলছে ({start_step_sft:,} -> {stage2_max_iters:,} স্টেপস)... ")
print("=" * 65)

sft_model.train()
optimizer.zero_grad(set_to_none=True)
start_time = time.time()

for step in range(start_step_sft, stage2_max_iters + 1):
    lr = get_sft_lr(step, stage2_max_iters, lr=GPTConfig.sft_learning_rate, min_lr=GPTConfig.sft_min_lr)
    for param_group in optimizer.param_groups: param_group['lr'] = lr

    accum_loss = 0.0
    for micro in range(GPTConfig.gradient_accumulation_steps):
        x, y = dataset_stage2.get_batch('train', batch_size=GPTConfig.batch_size, device=device)
        with autocast(dtype=torch.float16):
            _, loss = sft_model(x, targets=y)
            loss = loss / GPTConfig.gradient_accumulation_steps
        scaler.scale(loss).backward()
        accum_loss += loss.item()

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(raw_sft.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)

    # প্রতি ২৫ স্টেপ পরপর লাইভ লস মনিটরিং
    if step % 25 == 0 or step == start_step_sft or step == stage2_max_iters:
        ep = (step * tokens_per_step) / dataset_stage2.train_len
        elapsed = time.time() - start_time
        spd = (step - start_step_sft + 1) / elapsed if elapsed > 0 else 0
        print(f"[Stage 2] Step {step:5d}/{stage2_max_iters} (Epoch {ep:.2f}) | Loss: {accum_loss:.4f} | LR: {lr:.2e} | Speed: {spd:.2f} it/s")

    if step % 250 == 0 or step == stage2_max_iters:
        ckpt = os.path.join(STAGE2_DIR, f"stage2_step_{step}.pt")
        torch.save(raw_sft.state_dict(), ckpt)

# চূড়ান্ত প্রোডাকশন মডেল সেভ
final_model_path = os.path.join(DRIVE_BASE_DIR, "checkpoint_stage_2_final.pt")
torch.save(raw_sft.state_dict(), final_model_path)
print(f"🎉 অভিনন্দন! চূড়ান্ত ২-স্টেজ প্রোডাকশন মডেল প্রস্তুত: {final_model_path}")

In [ ]:
# Step 8: 💬 চ্যাটবট টেস্ট সেল (প্রশ্ন করুন, মডেল উত্তর দেবে!)
final_model_path = os.path.join(DRIVE_BASE_DIR, "checkpoint_stage_2_final.pt")
chat_model = GPT(GPTConfig).to(device)
chat_model.load_state_dict(torch.load(final_model_path, map_location=device))
chat_model.eval()
tokenizer = Tokenizer.from_file("tokenizer.json")

# এখানে আপনার প্রশ্ন লিখুন:
user_question = "ডিজিটাল মার্কেটিং কী?"

prompt = f"প্রশ্ন: {user_question} উত্তর:"
enc = tokenizer.encode(prompt)
ids = enc.ids if hasattr(enc, 'ids') else enc
input_tensor = torch.tensor([ids], dtype=torch.long, device=device)

eos_id = tokenizer.token_to_id('<EOS>')

with torch.no_grad():
    out = chat_model.generate(
        input_tensor,
        max_new_tokens=150,
        temperature=0.7,
        top_k=40,
        repetition_penalty=1.25,
        eos_id=eos_id
    )

raw_text = tokenizer.decode(out[0].cpu().tolist())
# উত্তর শেষ হলে অতিরিক্ত টেক্সট কেটে দেওয়া
clean_answer = raw_text.split('<EOS>')[0].strip()

print("=" * 60)
print(clean_answer)
print("=" * 60)